In [1]:
import sys
sys.path.append("../src")  # to import your modules
from extract import load_competitions, load_matches, load_all_events
from transform import extract_shots_features, calculate_player_stats, get_undervalued_players

In [2]:
# Load available competitions
competitions = load_competitions()
print(competitions[["competition_id", "competition_name", "season_id", "season_name"]])

    competition_id        competition_name  season_id season_name
0                9           1. Bundesliga        281   2023/2024
1                9           1. Bundesliga         27   2015/2016
2             1267  African Cup of Nations        107        2023
3               16        Champions League          4   2018/2019
4               16        Champions League          1   2017/2018
..             ...                     ...        ...         ...
70              35      UEFA Europa League         75   1988/1989
71              53       UEFA Women's Euro        315        2025
72              53       UEFA Women's Euro        106        2022
73              72       Women's World Cup        107        2023
74              72       Women's World Cup         30        2019

[75 rows x 4 columns]


In [3]:
# Load matches - La Liga 20/21 season
matches = load_matches(competition_id=11, season_id=90)
print(f"Partidos cargados: {len(matches)}")
print(matches[["match_id", "home_team", "away_team", "match_date"]].head())

Partidos cargados: 35
   match_id                                          home_team  \
0   3773386  {'home_team_id': 206, 'home_team_name': 'Depor...   
1   3773565  {'home_team_id': 1049, 'home_team_name': 'Gran...   
2   3773457  {'home_team_id': 217, 'home_team_name': 'Barce...   
3   3773631  {'home_team_id': 218, 'home_team_name': 'Real ...   
4   3773665  {'home_team_id': 422, 'home_team_name': 'Osasu...   

                                           away_team  match_date  
0  {'away_team_id': 217, 'away_team_name': 'Barce...  2020-10-31  
1  {'away_team_id': 217, 'away_team_name': 'Barce...  2021-01-09  
2  {'away_team_id': 209, 'away_team_name': 'Celta...  2021-05-16  
3  {'away_team_id': 217, 'away_team_name': 'Barce...  2021-02-07  
4  {'away_team_id': 217, 'away_team_name': 'Barce...  2021-03-06  


In [4]:
# Load events from the first 20 matches
match_ids = matches["match_id"].head(20).tolist()
events = load_all_events(match_ids)

print(f"Total eventos: {len(events)}")
print(f"Tipos de evento: {events['type'].apply(lambda x: x['name']).value_counts().head(10)}")

Total eventos: 79763
Tipos de evento: type
Pass             23147
Ball Receipt*    22503
Carry            18803
Pressure          6281
Ball Recovery     1629
Duel               985
Block              742
Dribble            688
Clearance          607
Goal Keeper        568
Name: count, dtype: int64


In [5]:
# Filter shots only
shots = events[events["type"].apply(lambda x: x["name"]) == "Shot"]
print(f"Total disparos: {len(shots)}")
print(shots[["player", "team", "shot", "minute"]].head())

Total disparos: 484
                                                 player  \
574             {'id': 30756, 'name': 'Anssumane Fati'}   
680   {'id': 26387, 'name': 'Edgar Antonio Méndez Or...   
900           {'id': 5487, 'name': 'Antoine Griezmann'}   
928   {'id': 5503, 'name': 'Lionel Andrés Messi Cucc...   
1281  {'id': 24049, 'name': 'Luis Jesús Rioja Gonzál...   

                                         team  \
574          {'id': 217, 'name': 'Barcelona'}   
680   {'id': 206, 'name': 'Deportivo Alavés'}   
900          {'id': 217, 'name': 'Barcelona'}   
928          {'id': 217, 'name': 'Barcelona'}   
1281  {'id': 206, 'name': 'Deportivo Alavés'}   

                                                   shot  minute  
574   {'one_on_one': True, 'statsbomb_xg': 0.2009693...      12  
680   {'statsbomb_xg': 0.096383646, 'end_location': ...      16  
900   {'statsbomb_xg': 0.09887917, 'end_location': [...      19  
928   {'statsbomb_xg': 0.07893826, 'end_location': [...      22  


In [ ]:
# Step 1 - Clean and flatten shots data
clean_shots = extract_shots_features(shots)
print(f"Clean shots: {len(clean_shots)}")
print(clean_shots.head())

Clean shots: 484
      match_id                     player_name         team_name  minute  \
574    3773386                  Anssumane Fati         Barcelona      12   
680    3773386     Edgar Antonio Méndez Ortega  Deportivo Alavés      16   
900    3773386               Antoine Griezmann         Barcelona      19   
928    3773386  Lionel Andrés Messi Cuccittini         Barcelona      22   
1281   3773386       Luis Jesús Rioja González  Deportivo Alavés      30   

            xg  is_goal  
574   0.200969        0  
680   0.096384        0  
900   0.098879        0  
928   0.078938        0  
1281  0.976192        1  


In [ ]:
# Step 2 - Calculate player metrics
player_stats = calculate_player_stats(clean_shots, matches)
print(player_stats.head(10))

                        player_name         team_name   total_xg  total_shots  \
0             Santiago Mina Lorenzo        Celta Vigo   0.397082            2   
1               Sergio Ramos García       Real Madrid   1.135943            4   
2         Luis Jesús Rioja González  Deportivo Alavés   0.976192            1   
3                 Jaime Mata Arnaiz            Getafe   0.835227            2   
4        Maximiliano Gómez González          Valencia   0.817078            2   
5                Rafael Mir Vicente            Huesca   2.193185            6   
6            Borja Iglesias Quintas        Real Betis   0.543145            2   
7  Gonzalo Julián Melero Manzanares        Levante UD   0.404842            2   
8                 Víctor Ruíz Torre        Real Betis   0.286379            2   
9    Lionel Andrés Messi Cuccittini         Barcelona  12.009651          110   

   total_goals  matches_played  minutes  goals_p90  xg_p90  performance_score  
0            2              

In [ ]:
# Step 3 - Identify undervalued players
undervalued = get_undervalued_players(player_stats)
print(f"Undervalued players found: {len(undervalued)}")
print(undervalued.head(10))

Undervalued players found: 6
                       player_name  team_name   total_xg  total_shots  \
0   Lionel Andrés Messi Cuccittini  Barcelona  12.009651          110   
1                   Anssumane Fati  Barcelona   1.579674           12   
2  Ronald Federico Araújo da Silva  Barcelona   1.310489            6   
3                Antoine Griezmann  Barcelona   3.931066           30   
4             Pedro González López  Barcelona   2.631845           13   
5        Philippe Coutinho Correia  Barcelona   1.746298           19   

   total_goals  matches_played  minutes  goals_p90  xg_p90  performance_score  
0           13              20     1800       0.65    0.60              0.625  
1            4               6      540       0.67    0.26              0.465  
2            1               4      360       0.25    0.33              0.290  
3            5              16     1440       0.31    0.25              0.280  
4            1               8      720       0.12    0.33 

In [ ]:
# Step 4 - Load data into SQLite database
from load import create_tables, load_to_db, query_db

# Create schema
create_tables()

Tables created successfully


In [10]:
# Load processed data into the database
load_to_db(clean_shots, "shots")
load_to_db(player_stats, "player_stats")
load_to_db(undervalued, "undervalued_players")

Loaded 484 rows into table 'shots'.
Loaded 120 rows into table 'player_stats'.
Loaded 6 rows into table 'undervalued_players'.


In [ ]:
# Step 5 - Verify data with SQL queries
top_players = query_db("""
    SELECT player_name, team_name, total_shots, total_goals, xg_p90, performance_score
    FROM player_stats
    ORDER BY performance_score DESC
    LIMIT 10
""")
print(top_players)

                        player_name         team_name  total_shots  \
0             Santiago Mina Lorenzo        Celta Vigo            2   
1               Sergio Ramos García       Real Madrid            4   
2         Luis Jesús Rioja González  Deportivo Alavés            1   
3                 Jaime Mata Arnaiz            Getafe            2   
4        Maximiliano Gómez González          Valencia            2   
5                Rafael Mir Vicente            Huesca            6   
6            Borja Iglesias Quintas        Real Betis            2   
7  Gonzalo Julián Melero Manzanares        Levante UD            2   
8                 Víctor Ruíz Torre        Real Betis            2   
9    Lionel Andrés Messi Cuccittini         Barcelona          110   

   total_goals  xg_p90  performance_score  
0            2    0.40              1.200  
1            1    1.14              1.070  
2            1    0.98              0.990  
3            1    0.84              0.920  
4        

In [12]:
# Query undervalued players directly from the database
undervalued_db = query_db("""
    SELECT player_name, team_name, xg_p90, goals_p90, performance_score
    FROM undervalued_players
    ORDER BY xg_p90 DESC
""")
print(undervalued_db)

                       player_name  team_name  xg_p90  goals_p90  \
0   Lionel Andrés Messi Cuccittini  Barcelona    0.60       0.65   
1  Ronald Federico Araújo da Silva  Barcelona    0.33       0.25   
2             Pedro González López  Barcelona    0.33       0.12   
3                   Anssumane Fati  Barcelona    0.26       0.67   
4                Antoine Griezmann  Barcelona    0.25       0.31   
5        Philippe Coutinho Correia  Barcelona    0.25       0.14   

   performance_score  
0              0.625  
1              0.290  
2              0.225  
3              0.465  
4              0.280  
5              0.195  
